In [ ]:
# Run this cell to set the working directory to the repo root.
from pathlib import Path
import os, sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "method.py").exists() and (p / "tools").is_dir())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))


# AI-READI Naive Thresholding Comparison

This notebook constructs pooled-quantile naive thresholds on the fixed AI-READI cohort,
converts each subject's glucose trajectory into TIR-style compositions, and compares
naive thresholding against the settled DE baselines using the same linear-model style
as the existing real-data notebook.

In [1]:
import importlib.util
import subprocess
import sys

from tools.r_tools import ensure_r_packages, setup_r_environment

setup_r_environment()

if importlib.util.find_spec("rpy2") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "rpy2"])

ensure_r_packages(
    ["clarkeTest", "fdapace", "remotes", "WR"],
    github_packages={"WR": "yqgchen/WR"},
)

Using R installation at: C:\Program Files\R\R-4.4.1


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from statsmodels.formula.api import ols

from tools.processing_aireadi import format_response_name, load_ai_readi_cohort
from tools.downstream_tools import (
    build_tir_modeling_data,
    clarke_test,
    compute_quantile_matrix,
    fit_wr_scalar_model,
    pooled_quantile_thresholds,
    r_squared,
    tir_column_names,
)

REPO_ROOT = Path.cwd()

CONSENSUS_THRESHOLD_SETS = {
    "consensus_k2": [70, 181],
    "consensus_k4": [54, 70, 181, 251],
}

DE_THRESHOLD_SETS = {
    "de_k2": [96, 170],
    "de_k4": [90, 128, 172, 232],
}

RESPONSES = ["hdl_c", "log_triglycerides", "AIP", "hba1c"]

## Recreate the current AI-READI analytic cohort

The cohort below should match the fixed 573-subject cohort used in the current real-data comparison workflow.

In [3]:
filtered_data = load_ai_readi_cohort(REPO_ROOT)

print("Number of subjects:", filtered_data.shape[0])
print("HbA1c range:", filtered_data["hba1c"].min(), "--", filtered_data["hba1c"].max())

filtered_data[["id", "hdl_c", "log_triglycerides", "AIP", "hba1c"]].head()

Number of subjects: 573
HbA1c range: 4.5 -- 7.4


,id,hdl_c,log_triglycerides,AIP,hba1c
0,1001,92.0,4.418841,-0.102948,5.7
1,1002,41.0,5.030438,1.316866,5.6
2,1006,91.0,4.276666,-0.234193,5.4
3,1008,62.0,4.488636,0.361502,6.0
4,1011,41.0,5.924256,2.210684,6.7


## Pooled-quantile naive thresholds

Here `K` denotes the number of internal thresholds.

- `K = 2` uses pooled tertile cutoffs
- `K = 4` uses pooled quintile cutoffs

In [4]:
def format_integer_cutoffs(cutoffs) -> str:
    return ", ".join(str(int(round(float(c)))) for c in cutoffs)


In [5]:
naive_threshold_specs = {
    "naive_k2": 2,
    "naive_k4": 4,
}

naive_threshold_map = {}

for label, threshold_count in naive_threshold_specs.items():
    probs, cutoffs = pooled_quantile_thresholds(filtered_data["gl"], threshold_count)
    naive_threshold_map[label] = cutoffs.tolist()

threshold_display_specs = [
    (2, "Consensus", CONSENSUS_THRESHOLD_SETS["consensus_k2"]),
    (2, "DE", DE_THRESHOLD_SETS["de_k2"]),
    (2, "Naive", naive_threshold_map["naive_k2"]),
    (4, "Consensus", CONSENSUS_THRESHOLD_SETS["consensus_k4"]),
    (4, "DE", DE_THRESHOLD_SETS["de_k4"]),
    (4, "Naive", naive_threshold_map["naive_k4"]),
]

naive_thresholds_df = pd.DataFrame(
    [
        {"K": k, "Method": method, "Cutoffs": format_integer_cutoffs(cutoffs)}
        for k, method, cutoffs in threshold_display_specs
    ]
).set_index(["K", "Method"])
display(naive_thresholds_df)

Cutoffs
K Method                       
2 Consensus             70, 181
  DE                    96, 170
  Naive                109, 129
4 Consensus    54, 70, 181, 251
  DE          90, 128, 172, 232
  Naive      102, 113, 124, 141

## Subject-level TIR summaries

The notebook now adds both the settled DE threshold features and the pooled-quantile naive threshold features to the same subject-level dataframe.

In [6]:
threshold_sets = {
    **CONSENSUS_THRESHOLD_SETS,
    **DE_THRESHOLD_SETS,
    **naive_threshold_map,
}

modeling_data, feature_map = build_tir_modeling_data(filtered_data, threshold_sets, drop_gl=True)

feature_summary_df = pd.DataFrame(
    {
        "Predictor set": list(feature_map.keys()),
        "Columns used": [", ".join(cols[:-1]) for cols in feature_map.values()],
        "Dropped last bin": [cols[-1] for cols in feature_map.values()],
    }
).set_index("Predictor set")

display(Markdown("#### Predictor sets used in the linear models"))
display(feature_summary_df)

#### Predictor sets used in the linear models

,Columns used,Dropped last bin
Predictor set,,
consensus_k2,"TIR_40_69, TIR_70_180",TIR_181_400
consensus_k4,"TIR_40_53, TIR_54_69, TIR_70_180, TIR_181_250",TIR_251_400
de_k2,"TIR_40_95, TIR_96_169",TIR_170_400
de_k4,"TIR_40_89, TIR_90_127, TIR_128_171, TIR_172_231",TIR_232_400
naive_k2,"TIR_40_108, TIR_109_128",TIR_129_400
naive_k4,"TIR_40_101, TIR_102_112, TIR_113_123, TIR_124_140",TIR_141_400


## Linear-model comparisons

Within each `K`, the table reports consensus, naive, and DE fits side by side, with consensus shown first to mirror the reference notebook's comparison framing.

Each `AIC` entry now includes the matching Clarke test p-value in parentheses, and the final `HbA1c` row serves as a glucose-linked baseline outcome alongside the lipid outcomes. The right-most WR column still reports the fitted $R^2$ from Wasserstein regression.

In [8]:
probs = np.linspace(0.0, 1.0, 101)
qf_matrix = compute_quantile_matrix(filtered_data["gl"], probs)

comparison_specs = [
    {
        "comparison": "K=2",
        "display_suffix": "(K=2)",
        "de_key": "de_k2",
        "comparators": [
            ("Consensus", "consensus_k2"),
            ("Naive", "naive_k2"),
        ],
    },
    {
        "comparison": "K=4",
        "display_suffix": "(K=4)",
        "de_key": "de_k4",
        "comparators": [
            ("Consensus", "consensus_k4"),
            ("Naive", "naive_k4"),
        ],
    },
]


def format_aic_with_p_value(delta_aic, p_value):
    return f"{delta_aic:.1f} ({p_value:.4f})"


naive_comparison_rows = []
naive_display_df = pd.DataFrame(index=[format_response_name(response) for response in RESPONSES])
model_store = {}

for response in RESPONSES:
    response_label = format_response_name(response)
    y = modeling_data[response].to_numpy(dtype=float)
    wr_fit = fit_wr_scalar_model(y, qf_matrix, probs)
    wr_r2 = r_squared(y, wr_fit["fitted"])
    naive_display_df.at[response_label, "R^2 (WR)"] = f"{wr_r2:.3f}"

    for spec in comparison_specs:
        suffix = spec["display_suffix"]
        de_key = spec["de_key"]
        de_vars = feature_map[de_key][:-1]
        de_formula = f"{response} ~ " + " + ".join(de_vars)
        de_model = ols(de_formula, data=modeling_data).fit()

        naive_display_df.at[response_label, f"R^2 (DE) {suffix}"] = f"{de_model.rsquared:.3f}"

        pairwise_store = {}
        for comparator_label, comparator_key in spec["comparators"]:
            comparator_vars = feature_map[comparator_key][:-1]
            comparator_formula = f"{response} ~ " + " + ".join(comparator_vars)
            comparator_model = ols(comparator_formula, data=modeling_data).fit()

            clarke_details = clarke_test(
                modeling_data,
                response,
                comparator_vars,
                de_vars,
                return_details=True,
                model_a_label=comparator_label,
                model_b_label="DE",
            )
            delta_aic = float(comparator_model.aic - de_model.aic)

            naive_comparison_rows.append(
                {
                    "Response": response_label,
                    "Comparison": spec["comparison"],
                    "Comparator": comparator_label,
                    "Comparator key": comparator_key,
                    "DE key": de_key,
                    "R^2 comparator": float(comparator_model.rsquared),
                    "R^2 DE": float(de_model.rsquared),
                    "AIC comparator": float(comparator_model.aic),
                    "AIC DE": float(de_model.aic),
                    "AIC (Comparator - DE)": delta_aic,
                    "R^2 WR": wr_r2,
                    "Clarke stat": clarke_details["stat"],
                    "Clarke nobs": clarke_details["nobs"],
                    "Clarke preferred": clarke_details["preferred_model"] or "Tie",
                    "Clarke preferred wins": clarke_details["preferred_wins"],
                    "Clarke wins comparator": clarke_details["wins_model_a"],
                    "Clarke wins DE": clarke_details["wins_model_b"],
                    "p-value": clarke_details["p_value"],
                }
            )

            naive_display_df.at[response_label, f"R^2 ({comparator_label}) {suffix}"] = f"{comparator_model.rsquared:.3f}"
            naive_display_df.at[response_label, f"AIC ({comparator_label} - DE) {suffix}"] = format_aic_with_p_value(
                delta_aic,
                clarke_details["p_value"],
            )

            pairwise_store[comparator_label.lower()] = {
                "model": comparator_model,
                "vars": comparator_vars,
                "clarke": clarke_details,
                "delta_aic": delta_aic,
            }

        model_store[(response, spec["comparison"])] = {
            "de": de_model,
            "de_vars": de_vars,
            "wr": wr_fit,
            "pairwise": pairwise_store,
        }

naive_comparison_df = pd.DataFrame(naive_comparison_rows)
naive_numeric_df = naive_comparison_df.copy()

ordered_columns = []
for spec in comparison_specs:
    suffix = spec["display_suffix"]
    ordered_columns.extend(
        [
            f"R^2 (Consensus) {suffix}",
            f"R^2 (DE) {suffix}",
            f"R^2 (Naive) {suffix}",
            f"AIC (Consensus - DE) {suffix}",
            f"AIC (Naive - DE) {suffix}",
        ]
    )
ordered_columns.append("R^2 (WR)")

naive_display_df = naive_display_df.loc[:, ordered_columns]

# display(Markdown("#### Long-form comparison dataframe"))
# display(naive_comparison_df)

# display(Markdown("#### Final display table"))
display(naive_thresholds_df)
display(naive_display_df)

Cutoffs
K Method                       
2 Consensus             70, 181
  DE                    96, 170
  Naive                109, 129
4 Consensus    54, 70, 181, 251
  DE          90, 128, 172, 232
  Naive      102, 113, 124, 141

,R^2 (Consensus) (K=2),R^2 (DE) (K=2),R^2 (Naive) (K=2),AIC (Consensus - DE) (K=2),AIC (Naive - DE) (K=2),R^2 (Consensus) (K=4),R^2 (DE) (K=4),R^2 (Naive) (K=4),AIC (Consensus - DE) (K=4),AIC (Naive - DE) (K=4),R^2 (WR)
HDL-C,0.011,0.018,0.035,4.2 (0.0001),-10.3 (0.0000),0.014,0.041,0.040,16.2 (0.0000),0.9 (0.4035),0.041
TG,0.025,0.049,0.050,14.5 (0.0000),-0.8 (0.8673),0.040,0.052,0.053,7.5 (0.0366),-0.3 (0.4035),0.054
TG/HDL-C,0.026,0.050,0.059,14.3 (0.0000),-5.5 (0.0095),0.040,0.060,0.062,11.8 (0.0058),-1.3 (0.0026),0.067
HbA1c,0.217,0.225,0.208,5.6 (0.0297),12.5 (0.0192),0.227,0.248,0.231,15.9 (0.0001),13.4 (0.0001),0.255


In [10]:
from pathlib import Path

output_dir = Path("results/tables")
output_dir.mkdir(parents=True, exist_ok=True)

RESPONSE_ORDER = ["HDL-C", "TG", "TG/HDL-C", "HbA1c"]


def _significance_star(p_value):
    """Return a LaTeX significance marker based on the p-value."""
    if p_value < 0.001:
        return r"$^\ddagger$"
    if p_value < 0.01:
        return r"$^\dagger$"
    if p_value < 0.05:
        return r"$^*$"
    return ""


def naive_display_to_latex(
    naive_display_df,
    naive_numeric_df,
    *,
    label="tab:naive_lm_results",
):
    """
    Build a LaTeX table matching the main.tex tab:lm_results format,
    but comparing DE vs Naive instead of DE vs Consensus.
    """
    lines = [
        r"\begin{table}[t]",
        r"\centering",
        r"\caption{Comparison of linear model fits with TIR compositional predictors "
        r"based on data-driven (DE) and naive pooled-quantile thresholds. "
        r"$\Delta$AIC denotes the difference AIC$_{\text{Naive}}$ - AIC$_{\text{DE}}$. "
        r"Significant $p$-values from Clarke's test for non-nested model comparison "
        r"are indicated with $\Delta$AIC. "
        r"The WR column reports the $R^2$ from Wasserstein regression, "
        r"which uses the full quantile function as the predictor without thresholding.}",
        rf"\label{{{label}}}",
        r"\small",
        r"\begin{tabular}{lrrrrrrr}",
        r"\toprule",
        r" & \multicolumn{3}{c}{$K=2$} & \multicolumn{3}{c}{$K=4$} & \multicolumn{1}{c}{WR} \\",
        r"\cmidrule(lr){2-4} \cmidrule(lr){5-7} \cmidrule(lr){8-8}"
        r" Response"
        r" & \multicolumn{1}{c}{$R^2_{\text{DE}}$}"
        r" & \multicolumn{1}{c}{$R^2_{\text{Naive}}$}"
        r" & \multicolumn{1}{c}{$\Delta$AIC}"
        r" & \multicolumn{1}{c}{$R^2_{\text{DE}}$}"
        r" & \multicolumn{1}{c}{$R^2_{\text{Naive}}$}"
        r" & \multicolumn{1}{c}{$\Delta$AIC}"
        r" & \multicolumn{1}{c}{$R^2$} \\",
        r"\midrule",
    ]

    for response_label in RESPONSE_ORDER:
        if response_label not in naive_display_df.index:
            continue

        row = naive_display_df.loc[response_label]
        parts = [response_label]

        for k_suffix in ["(K=2)", "(K=4)"]:
            r2_de = row[f"R^2 (DE) {k_suffix}"]
            r2_nv = row[f"R^2 (Naive) {k_suffix}"]

            # Extract numeric ΔAIC and p-value from naive_numeric_df
            k_label = k_suffix.strip("()")  # "K=2" or "K=4"
            match = naive_numeric_df[
                (naive_numeric_df["Response"] == response_label)
                & (naive_numeric_df["Comparison"] == k_label)
                & (naive_numeric_df["Comparator"] == "Naive")
            ]
            delta_aic = float(match["AIC (Comparator - DE)"].iloc[0])
            p_value = float(match["p-value"].iloc[0])
            star = _significance_star(p_value)

            parts.append(r2_de)
            parts.append(r2_nv)
            parts.append(f"{delta_aic:.1f}{star}")

        parts.append(row["R^2 (WR)"])
        lines.append(" & ".join(parts) + r" \\")

    lines.extend([
        r"\bottomrule",
        r"\end{tabular}",
        r"\begin{tablenotes}",
        r"\footnotesize",
        r"\item \hspace{7em} * $p<0.05$, † $p<0.01$, ‡ $p<0.001$",
        r"\end{tablenotes}",
        r"\end{table}",
    ])
    return "\n".join(lines)


naive_lm_tex = naive_display_to_latex(naive_display_df, naive_numeric_df)

tex_path = output_dir / "aireadi_naive_lm_results.tex"
tex_path.write_text(naive_lm_tex, encoding="utf-8")

print(f"Exported: {tex_path}")
print()
print(naive_lm_tex)

Exported: results\tables\aireadi_naive_lm_results.tex

\begin{table}[t]
\centering
\caption{Comparison of linear model fits with TIR compositional predictors based on data-driven (DE) and naive pooled-quantile thresholds. $\Delta$AIC denotes the difference AIC$_{\text{Naive}}$ - AIC$_{\text{DE}}$. Significant $p$-values from Clarke's test for non-nested model comparison are indicated with $\Delta$AIC. The WR column reports the $R^2$ from Wasserstein regression, which uses the full quantile function as the predictor without thresholding.}
\label{tab:naive_lm_results}
\small
\begin{tabular}{lrrrrrrr}
\toprule
 & \multicolumn{3}{c}{$K=2$} & \multicolumn{3}{c}{$K=4$} & \multicolumn{1}{c}{WR} \\
\cmidrule(lr){2-4} \cmidrule(lr){5-7} \cmidrule(lr){8-8} Response & \multicolumn{1}{c}{$R^2_{\text{DE}}$} & \multicolumn{1}{c}{$R^2_{\text{Naive}}$} & \multicolumn{1}{c}{$\Delta$AIC} & \multicolumn{1}{c}{$R^2_{\text{DE}}$} & \multicolumn{1}{c}{$R^2_{\text{Naive}}$} & \multicolumn{1}{c}{$\Delta$AIC} 

In [14]:
def full_comparison_transposed_to_latex(
    naive_display_df,
    naive_numeric_df,
    *,
    response_subset=None,
    label="tab:full_lm_results",
    caption=None,
):
    """
    Transposed LaTeX table: rows = metrics grouped by K, columns = responses.
    Style follows the Kendall tau table in main.tex.

    Parameters
    ----------
    response_subset : list[str] | None
        Subset of RESPONSE_ORDER to include.  ``None`` → all responses.
    """
    if response_subset is None:
        response_subset = RESPONSE_ORDER
    responses = [r for r in response_subset if r in naive_display_df.index]
    n_resp = len(responses)
    col_spec = "ll" + "r" * n_resp

    resp_cols = " & ".join(rf"\multicolumn{{1}}{{c}}{{{r}}}" for r in responses)

    if caption is None:
        caption = (
            r"Comparison of linear model fits with TIR compositional predictors "
            r"based on consensus (CS), data-driven (DE), and naive pooled-quantile thresholds. "
            r"$\Delta$AIC denotes the difference in AIC relative to DE: "
            r"$\Delta$AIC(CS) $=$ AIC$_{\text{CS}}$ $-$ AIC$_{\text{DE}}$ and "
            r"$\Delta$AIC(Naive) $=$ AIC$_{\text{Naive}}$ $-$ AIC$_{\text{DE}}$. "
            r"Significant $p$-values from Clarke's test for non-nested model comparison "
            r"are indicated with $\Delta$AIC."
        )

    lines = [
        r"\begin{table}[t]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\small",
        r"\begin{tabular}{" + col_spec + "}",
        r"\toprule",
        rf"$K$ & Metric & {resp_cols} \\",
        r"\midrule",
    ]

    METRIC_ROWS = [
        ("R^2_CS", r"$R^2_{\text{CS}}$", "R^2 (Consensus)"),
        ("R^2_DE", r"$R^2_{\text{DE}}$", "R^2 (DE)"),
        ("R^2_Naive", r"$R^2_{\text{Naive}}$", "R^2 (Naive)"),
        ("AIC_CS", r"$\Delta$AIC (CS$-$DE)", "Consensus"),
        ("AIC_Naive", r"$\Delta$AIC (Naive$-$DE)", "Naive"),
    ]

    for k_label, k_suffix in [("$K=2$", "(K=2)"), ("$K=4$", "(K=4)")]:
        for metric_idx, (metric_key, metric_tex, df_key) in enumerate(METRIC_ROWS):
            if metric_idx == 0:
                group_cell = rf"\multirow{{5}}{{*}}{{{k_label}}}"
            else:
                group_cell = ""

            values = []
            for resp in responses:
                row = naive_display_df.loc[resp]
                if metric_key.startswith("R^2"):
                    values.append(row[f"{df_key} {k_suffix}"])
                else:
                    k_comp = k_suffix.strip("()")
                    match = naive_numeric_df[
                        (naive_numeric_df["Response"] == resp)
                        & (naive_numeric_df["Comparison"] == k_comp)
                        & (naive_numeric_df["Comparator"] == df_key)
                    ]
                    delta = float(match["AIC (Comparator - DE)"].iloc[0])
                    p_val = float(match["p-value"].iloc[0])
                    values.append(f"{delta:.1f}{_significance_star(p_val)}")

            line = " & ".join([group_cell, metric_tex] + values) + r" \\"
            lines.append(line)

        lines.append(r"\midrule")

    # WR row — full distribution, no thresholding
    wr_values = [naive_display_df.loc[resp, "R^2 (WR)"] for resp in responses]
    wr_line = " & ".join([r"Full dist.", r"$R^2_{\text{WR}}$"] + wr_values) + r" \\"
    lines.append(wr_line)

    lines.extend([
        r"\bottomrule",
        r"\end{tabular}",
        r"\begin{tablenotes}",
        r"\footnotesize",
        r"\item \hspace{7em} * $p<0.05$, † $p<0.01$, ‡ $p<0.001$",
        r"\end{tablenotes}",
        r"\end{table}",
    ])
    return "\n".join(lines)


# --- Lipid responses only (exclude HbA1c) ---
LIPID_RESPONSES = ["HDL-C", "TG", "TG/HDL-C"]

full_tex = full_comparison_transposed_to_latex(
    naive_display_df,
    naive_numeric_df,
    response_subset=LIPID_RESPONSES,
    label="tab:full_lm_results",
)

full_tex_path = output_dir / "aireadi_full_lm_results.tex"
full_tex_path.write_text(full_tex, encoding="utf-8")

print(f"Exported: {full_tex_path}")
print()
print(full_tex)

Exported: results\tables\aireadi_full_lm_results.tex

\begin{table}[t]
\centering
\caption{Comparison of linear model fits with TIR compositional predictors based on consensus (CS), data-driven (DE), and naive pooled-quantile thresholds. $\Delta$AIC denotes the difference in AIC relative to DE: $\Delta$AIC(CS) $=$ AIC$_{\text{CS}}$ $-$ AIC$_{\text{DE}}$ and $\Delta$AIC(Naive) $=$ AIC$_{\text{Naive}}$ $-$ AIC$_{\text{DE}}$. Significant $p$-values from Clarke's test for non-nested model comparison are indicated with $\Delta$AIC.}
\label{tab:full_lm_results}
\small
\begin{tabular}{llrrr}
\toprule
$K$ & Metric & \multicolumn{1}{c}{HDL-C} & \multicolumn{1}{c}{TG} & \multicolumn{1}{c}{TG/HDL-C} \\
\midrule
\multirow{5}{*}{$K=2$} & $R^2_{\text{CS}}$ & 0.011 & 0.025 & 0.026 \\
 & $R^2_{\text{DE}}$ & 0.018 & 0.049 & 0.050 \\
 & $R^2_{\text{Naive}}$ & 0.035 & 0.050 & 0.059 \\
 & $\Delta$AIC (CS$-$DE) & 4.2$^\ddagger$ & 14.5$^\ddagger$ & 14.3$^\ddagger$ \\
 & $\Delta$AIC (Naive$-$DE) & -10.3$^\dd

In [15]:
# --- HbA1c table (same format, single response column) ---
hba1c_tex = full_comparison_transposed_to_latex(
    naive_display_df,
    naive_numeric_df,
    response_subset=["HbA1c"],
    label="tab:full_lm_results_hba1c",
)

hba1c_tex_path = output_dir / "aireadi_full_lm_results_hba1c.tex"
hba1c_tex_path.write_text(hba1c_tex, encoding="utf-8")

print(f"Exported: {hba1c_tex_path}")
print()
print(hba1c_tex)

Exported: results\tables\aireadi_full_lm_results_hba1c.tex

\begin{table}[t]
\centering
\caption{Comparison of linear model fits with TIR compositional predictors based on consensus (CS), data-driven (DE), and naive pooled-quantile thresholds. $\Delta$AIC denotes the difference in AIC relative to DE: $\Delta$AIC(CS) $=$ AIC$_{\text{CS}}$ $-$ AIC$_{\text{DE}}$ and $\Delta$AIC(Naive) $=$ AIC$_{\text{Naive}}$ $-$ AIC$_{\text{DE}}$. Significant $p$-values from Clarke's test for non-nested model comparison are indicated with $\Delta$AIC.}
\label{tab:full_lm_results_hba1c}
\small
\begin{tabular}{llr}
\toprule
$K$ & Metric & \multicolumn{1}{c}{HbA1c} \\
\midrule
\multirow{5}{*}{$K=2$} & $R^2_{\text{CS}}$ & 0.217 \\
 & $R^2_{\text{DE}}$ & 0.225 \\
 & $R^2_{\text{Naive}}$ & 0.208 \\
 & $\Delta$AIC (CS$-$DE) & 5.6$^*$ \\
 & $\Delta$AIC (Naive$-$DE) & 12.5$^*$ \\
\midrule
\multirow{5}{*}{$K=4$} & $R^2_{\text{CS}}$ & 0.227 \\
 & $R^2_{\text{DE}}$ & 0.248 \\
 & $R^2_{\text{Naive}}$ & 0.231 \\
 & $\

## Kendall's tau checks


In [8]:
from scipy.stats import kendalltau

kendall_data = modeling_data.copy()
kendall_data["tg_hdl_c"] = kendall_data["triglycerides"] / kendall_data["hdl_c"]

kendall_targets = ["hdl_c", "ldl_c", "total_c", "triglycerides", "tg_hdl_c", "hba1c"]
target_label_map = {
    "hdl_c": "HDL-C",
    "ldl_c": "LDL-C",
    "total_c": "Total-C",
    "triglycerides": "TG",
    "tg_hdl_c": "TG/HDL-C",
    "hba1c": "HbA1c",
}

def format_kendall_p_value(p_value):
    return f"{float(p_value):.4f}".removeprefix("0")


def format_kendall_entry(tau, p_value):
    return f"{tau:.3f} ({format_kendall_p_value(p_value)})"


kendall_tables = {
    "K=2": pd.DataFrame(columns=feature_map["de_k2"] + feature_map["naive_k2"], index=kendall_targets),
    "K=4": pd.DataFrame(columns=feature_map["de_k4"] + feature_map["naive_k4"], index=kendall_targets),
}

for target in kendall_targets:
    x = kendall_data[target].to_numpy(dtype=float)
    for k_label, kendall_table in kendall_tables.items():
        for tir in kendall_table.columns:
            tau, p_value = kendalltau(x, kendall_data[tir].to_numpy(dtype=float))
            kendall_table.loc[target, tir] = format_kendall_entry(tau, p_value)

combined_kendall_rows = []
for k_label, method_key_pairs in {
    "K=2": [("DE", "de_k2"), ("Naive", "naive_k2")],
    "K=4": [("DE", "de_k4"), ("Naive", "naive_k4")],
}.items():
    kendall_table = kendall_tables[k_label]
    for method_label, feature_key in method_key_pairs:
        for tir in feature_map[feature_key]:
            row = {"K": k_label, "Method": method_label, "TIR": tir}
            for target in kendall_targets:
                row[target_label_map[target]] = kendall_table.loc[target, tir]
            combined_kendall_rows.append(row)

naive_kendall_df = pd.DataFrame(combined_kendall_rows)
display(naive_kendall_df)

,K,Method,TIR,HDL-C,LDL-C,Total-C,TG,TG/HDL-C,HbA1c
0,K=2,DE,TIR_40_95,0.088 (.0019),-0.013 (.6342),-0.017 (.5349),-0.113 (.0001),-0.119 (.0000),-0.219 (.0000)
1,K=2,DE,TIR_96_169,-0.027 (.3443),-0.002 (.9498),0.005 (.8633),0.058 (.0383),0.057 (.0431),-0.031 (.2786)
2,K=2,DE,TIR_170_400,-0.042 (.1396),-0.006 (.8279),0.013 (.6536),0.083 (.0030),0.079 (.0050),0.299 (.0000)
3,K=2,Naive,TIR_40_108,0.112 (.0001),-0.001 (.9763),-0.003 (.9182),-0.131 (.0000),-0.144 (.0000),-0.258 (.0000)
4,K=2,Naive,TIR_109_128,0.040 (.1514),-0.010 (.7138),0.007 (.8101),0.025 (.3754),0.001 (.9622),-0.101 (.0005)
5,K=2,Naive,TIR_129_400,-0.133 (.0000),0.003 (.9243),-0.002 (.9535),0.129 (.0000),0.151 (.0000),0.298 (.0000)
6,K=4,DE,TIR_40_89,0.081 (.0041),-0.011 (.7011),-0.015 (.5887),-0.107 (.0001),-0.112 (.0001),-0.207 (.0000)
7,K=4,DE,TIR_90_127,0.125 (.0000),0.001 (.9671),0.020 (.4795),-0.083 (.0029),-0.113 (.0001),-0.248 (.0000)
8,K=4,DE,TIR_128_171,-0.143 (.0000),0.003 (.9266),-0.006 (.8346),0.130 (.0000),0.156 (.0000),0.265 (.0000)
9,K=4,DE,TIR_172_231,-0.045 (.1143),-0.004 (.8767),0.013 (.6414),0.084 (.0027),0.080 (.0044),0.299 (.0000)


For Naive, TIR 113_123 is largely uninformative